# Analisis K-Fold Cross Validation pada Dataset Heart Disease

**Repositori**: Machine Learning  
**Topik**: Perbandingan Teknik Cross Validation pada Berbagai Model Klasifikasi  
**Dataset**: `heart_disease_data.csv`  

---

## Pendahuluan
**Cross Validation** adalah teknik resampling yang digunakan untuk mengevaluasi model machine learning dengan membagi data menjadi *k* subset (*fold*). Model dilatih pada *k-1* fold dan diuji pada 1 fold sisanya, kemudian diulang sebanyak *k* kali. Pendekatan ini menghasilkan estimasi performa yang lebih stabil dan tidak bias dibandingkan *single train/test split*.

Notebook ini membandingkan **K-Fold Cross Validation** (k=5 dan k=10), **StratifiedKFold**, serta **single train/test split** pada empat algoritma klasifikasi.

### Alur Kerja:
1. **Data Acquisition & Understanding**: Memuat dan mengeksplorasi dataset.
2. **Data Preparation**: Standarisasi fitur numerik.
3. **Modeling & Cross Validation**: Menerapkan K-Fold (k=5, 10), StratifiedKFold pada empat model.
4. **Comparison**: Membandingkan hasil CV dengan single train/test split.
5. **Insight**: Menentukan model paling stabil dan akurat.

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)

## 1. Data Acquisition
Memuat dataset heart disease dari file CSV.

In [ ]:
DATA_PATH = '../../data/heart_disease_data.csv'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'Dataset tidak ditemukan di {os.path.abspath(DATA_PATH)}')

df = pd.read_csv(DATA_PATH)
print(f'Dataset berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom')

## 2. Exploratory Data Analysis (EDA)
Menampilkan informasi dasar dataset, statistik deskriptif, distribusi target, dan matriks korelasi.

In [ ]:
print('--- 5 Baris Pertama ---')
display(df.head())

print('\n--- Informasi Dataset ---')
df.info()

print('\n--- Statistik Deskriptif ---')
display(df.describe())

print('\n--- Cek Missing Values ---')
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribusi target
sns.countplot(data=df, x='target', ax=axes[0])
axes[0].set_title('Distribusi Target (0 = Tidak Sakit, 1 = Sakit)', fontsize=13)
axes[0].set_xlabel('Target')
axes[0].set_ylabel('Jumlah')

# Distribusi usia
sns.histplot(data=df, x='age', bins=20, kde=True, ax=axes[1])
axes[1].set_title('Distribusi Usia Pasien', fontsize=13)
axes[1].set_xlabel('Usia')

# Korelasi heatmap
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu', center=0,
            square=True, ax=axes[2])
axes[2].set_title('Matriks Korelasi', fontsize=13)

plt.tight_layout()
plt.show()

## 3. Data Preparation
Dataset ini tidak memiliki missing values. Langkah persiapan meliputi pemisahan fitur (X) dan target (y), serta standarisasi fitur numerik menggunakan **StandardScaler**.

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

print(f'Fitur (X): {X.shape}')
print(f'Target (y): {y.value_counts().to_dict()}')

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print('Standarisasi selesai. Rata-rata tiap fitur (mendekati 0):')
print(np.round(X_scaled.mean(), 2).to_dict())

## 4. Definisi Model dan Fungsi Evaluasi
Empat model yang akan dibandingkan:
1. **Logistic Regression** — model linear untuk klasifikasi biner
2. **K-Nearest Neighbors (k=5)** — model berbasis jarak
3. **Decision Tree** — model berbasis pohon keputusan
4. **Random Forest (n_estimators=50)** — ensemble dari decision tree

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K-Nearest Neighbors (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest (n=50)': RandomForestClassifier(n_estimators=50, random_state=42)
}

cv_methods = {
    'KFold (k=5)': KFold(n_splits=5, shuffle=True, random_state=42),
    'KFold (k=10)': KFold(n_splits=10, shuffle=True, random_state=42),
    'StratifiedKFold (k=5)': StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'StratifiedKFold (k=10)': StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
}

print('Model dan metode CV siap digunakan.')

## 5. K-Fold Cross Validation — Semua Model & Metode
Menghitung skor akurasi untuk setiap kombinasi model dan metode CV.

In [ ]:
results = []

for model_name, model in models.items():
    for cv_name, cv in cv_methods.items():
        scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
        results.append({
            'Model': model_name,
            'CV Method': cv_name,
            'Scores': scores,
            'Mean Accuracy': np.mean(scores),
            'Std Dev': np.std(scores)
        })

results_df = pd.DataFrame(results)

print('\n=== Rata-rata Accuracy dan Std Dev ===')
display_df = results_df[['Model', 'CV Method', 'Mean Accuracy', 'Std Dev']].copy()
display_df['Mean Accuracy'] = display_df['Mean Accuracy'].round(4)
display_df['Std Dev'] = display_df['Std Dev'].round(4)
display(display_df)

## 6. Boxplot Perbandingan Skor CV
Visualisasi distribusi skor akurasi CV untuk setiap model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, model_name in enumerate(models.keys()):
    model_data = results_df[results_df['Model'] == model_name]
    data_to_plot = [scores for scores in model_data['Scores']]
    labels = model_data['CV Method'].tolist()

    bp = axes[i].boxplot(data_to_plot, labels=labels, patch_artist=True, showmeans=True)
    colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    axes[i].set_title(f'{model_name}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Accuracy')
    axes[i].set_ylim(0.4, 1.0)
    axes[i].tick_params(axis='x', rotation=25)

plt.suptitle('Perbandingan Skor Cross Validation Antar Model', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

model_order = list(models.keys())
for cv_name in cv_methods.keys():
    subset = results_df[results_df['CV Method'] == cv_name]
    means = [subset[subset['Model'] == m]['Mean Accuracy'].values[0] for m in model_order]
    stds = [subset[subset['Model'] == m]['Std Dev'].values[0] for m in model_order]
    ax.errorbar(model_order, means, yerr=stds, marker='o', capsize=5,
                markersize=8, linewidth=2, label=cv_name)

ax.set_title('Rata-rata Accuracy CV dengan Error Bar (Std Dev)', fontsize=14, fontweight='bold')
ax.set_ylabel('Mean Accuracy')
ax.set_ylim(0.5, 1.0)
ax.legend(loc='lower left', fontsize=10)
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 7. Perbandingan dengan Single Train/Test Split
Membandingkan hasil terbaik CV dengan pendekatan single train/test split (80:20). Perbedaan ini menunjukkan mengapa CV lebih dapat diandalkan.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

single_split_results = []

for model_name, model in models.items():
    clf = model.__class__(**model.get_params())
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    single_split_results.append({'Model': model_name, 'Single Split Accuracy': round(acc, 4)})

single_df = pd.DataFrame(single_split_results)

# Ambil rata-rata terbaik dari CV (KFold 10)
best_cv = results_df[results_df['CV Method'] == 'StratifiedKFold (k=10)']
best_cv = best_cv[['Model', 'Mean Accuracy', 'Std Dev']].copy()
best_cv['Mean Accuracy'] = best_cv['Mean Accuracy'].round(4)
best_cv['Std Dev'] = best_cv['Std Dev'].round(4)

comparison = single_df.merge(best_cv, on='Model')
comparison['Selisih'] = (comparison['Single Split Accuracy'] - comparison['Mean Accuracy']).round(4)

print('=== Perbandingan Single Split vs StratifiedKFold (k=10) ===')
display(comparison)

print('\nCatatan: Selisih positif artinya single split memberikan akurasi lebih tinggi dari CV.')
print('Namun, single split sangat bergantung pada komposisi acak data train/test, sehingga kurang stabil dibandingkan CV.')

## 8. Visualisasi Perbandingan: CV vs Single Split

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(comparison['Model']))
width = 0.3

bars1 = ax.bar(x - width/2, comparison['Single Split Accuracy'], width,
               label='Single Train/Test Split', color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x + width/2, comparison['Mean Accuracy'], width,
               label='StratifiedKFold (k=10) — Rata-rata CV', color='#C44E52', alpha=0.85)

ax.set_ylabel('Accuracy')
ax.set_title('Perbandingan Akurasi: Single Split vs Cross Validation', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'], rotation=20, ha='right')
ax.set_ylim(0.5, 1.0)
ax.legend(loc='lower right')

for bar1, bar2, row in zip(bars1, bars2, comparison.itertuples()):
    ax.text(bar1.get_x() + bar1.get_width()/2, bar1.get_height() + 0.01,
            f'{row._2:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(bar2.get_x() + bar2.get_width()/2, bar2.get_height() + 0.01,
            f'{row._3:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
print('=== Perbandingan Std Dev Antar Model (StratifiedKFold k=10) ===')
std_comparison = best_cv.sort_values('Std Dev')
display(std_comparison)

print('\nSemakin kecil Std Dev, semakin stabil model terhadap perubahan data training.')

## 9. Insight & Kesimpulan

Berdasarkan hasil analisis K-Fold Cross Validation pada dataset Heart Disease:

### Stabilitas Model (Variance Rendah):
- Model dengan **std dev paling kecil** menunjukkan performa yang konsisten di setiap fold — artinya model tidak terlalu sensitif terhadap perubahan data training.
- Random Forest cenderung memiliki variance rendah karena sifat *ensemble* yang merata-rata prediksi banyak pohon.
- Logistic Regression juga cukup stabil karena model linear yang sederhana.

### Akurasi Tertinggi:
- Model dengan **rata-rata accuracy tertinggi** di semua metode CV adalah model yang paling akurat secara general.
- Random Forest umumnya unggul dalam akurasi karena mampu menangkap pola non-linear dan interaksi antar fitur.

### K-Fold vs StratifiedKFold:
- **StratifiedKFold** mempertahankan proporsi kelas target di setiap fold, sehingga menghasilkan estimasi yang lebih representatif untuk dataset dengan kelas tidak seimbang.
- **K-Fold biasa** bisa menghasilkan fold dengan distribusi kelas yang timpang jika dataset tidak seimbang.

### Kelebihan Cross Validation vs Single Split:
- Cross Validation memanfaatkan seluruh data untuk training dan testing, menghindari *lucky/unlucky split*.
- Single split bergantung pada satu komposisi train/test — hasilnya bisa sangat bervariasi.
- CV memberikan estimasi performa yang lebih *reliable* dan *generalizable*.

### Ringkasan:
> **Model terbaik** adalah yang memiliki **akurasi tinggi** dan **std dev rendah**. Jika harus memilih satu, **Random Forest dengan StratifiedKFold (k=10)** memberikan keseimbangan terbaik antara akurasi dan stabilitas untuk dataset ini. **Logistic Regression** bisa menjadi alternatif yang lebih sederhana dan *interpretable* dengan performa yang kompetitif.